In [10]:
import pandas as pd
from datasets import DatasetDict, load_dataset
from langchain_huggingface import HuggingFacePipeline
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
)
from langchain_chroma import Chroma

In [2]:
tmp_data = pd.read_json("ori_pqal.json").T
# some labels have been defined as "maybe", only keep the yes/no answers
tmp_data = tmp_data[tmp_data.final_decision.isin(["yes", "no"])]

documents = pd.DataFrame({"abstract": tmp_data.apply(lambda row: (" ").join(row.CONTEXTS+[row.LONG_ANSWER]), axis=1),
             "year": tmp_data.YEAR})
questions = pd.DataFrame({"question": tmp_data.QUESTION,
             "year": tmp_data.YEAR,
             "gold_label": tmp_data.final_decision,
             "gold_context": tmp_data.LONG_ANSWER,
             "gold_document_id": documents.index})

In [3]:
documents.head()

,abstract,year
21645374,Programmed cell death (PCD) is the regulated d...,2011
16418930,Assessment of visual acuity depends on the opt...,2006
9488747,Apparent life-threatening events in infants ar...,1997
17208539,The transanal endorectal pull-through (TERPT) ...,2007
10808977,Telephone counseling and tailored print commun...,2000


In [4]:
questions.head()

,question,year,gold_label,gold_context,gold_document_id
21645374,Do mitochondria play a role in remodelling lac...,2011,yes,Results depicted mitochondrial dynamics in viv...,21645374
16418930,Landolt C and snellen e acuity: differences in...,2006,no,"Using the charts described, there was only a s...",16418930
9488747,"Syncope during bathing in infants, a pediatric...",1997,yes,"""Aquagenic maladies"" could be a pediatric form...",9488747
17208539,Are the long-term results of the transanal pul...,2007,no,Our long-term study showed significantly bette...,17208539
10808977,Can tailored interventions increase mammograph...,2000,yes,The effects of the intervention were most pron...,10808977


In [5]:
questions.iloc[0].question

'Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?'

In [6]:
documents.iloc[0].abstract

'Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in late stages of PCD (LPCD). Window stage leaves were stained with the mitochondrial dye MitoTracker Red CMXRos a

In [7]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    encode_kwargs={"normalize_embeddings": True},
)

In [18]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    add_start_index=True
)

metadatas = [{"id": idx} for idx in documents.index]
texts = text_splitter.create_documents(
    texts=documents.abstract.tolist(),
    metadatas=metadatas,
)

print(len(documents), "docs ->", len(texts), "chunks")
for t in texts[:3]:
    print(t.metadata, t.page_content[:], "\n---")

890 docs -> 1902 chunks
{'id': 21645374, 'start_index': 0} Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in late stages of PCD (LPCD). Window stage leaves were s

In [20]:
vector_store.delete_collection()
vector_store = Chroma.from_documents(
    documents=texts[:15],
    embedding=embeddings,
    collection_name="documents",
    collection_metadata={"hnsw:space": "cosine"},
)

In [12]:
results = vector_store.similarity_search_with_score("What is programmed cell death?", k=3)
for res, score in results:
    print(f"* [DIST={score:3f}] {res.page_content} [{res.metadata}]")

* [DIST=0.575819] Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in late stages of PCD (LPCD). Window stage leaves were stained with the mitochondrial dye MitoTra

In [13]:
model = HuggingFacePipeline.from_model_id(
    model_id="meta-llama/Llama-3.2-3B-Instruct",
    task="text-generation",
    model_kwargs={"torch_dtype": "auto"},
    pipeline_kwargs={"max_new_tokens": 256, "return_full_text": False},
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cpu


In [23]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# 1. Retriever: start with k=1, as the assignment suggests
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# 2. Prompt
template = """You are a biomedical expert. Answer the question using only the context below.
Answer with a single word: "Yes" or "No".

Context:
{context}

Question: {question}

Answer:"""
prompt = ChatPromptTemplate.from_template(template)

# 3. Turn the retrieved Documents into a single string for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 4. Fetch the context and pass the question through, in parallel
runnable_parallel_object = RunnableParallel(
    context=retriever,               # -> list[Document] (metadata kept)
    question=RunnablePassthrough(),  # -> the raw question string
)

# 5. Generation chain: formats the context as text, then prompt -> LM -> string
chain = (
    RunnablePassthrough.assign(context=lambda x: format_docs(x["context"]))
    | prompt
    | model
    | StrOutputParser()
)

# 6. Combine: keep the retrieved docs and add the generated answer
rag_chain = runnable_parallel_object.assign(answer=chain)

In [48]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

template = """You are a biomedical expert. Answer the question using only the context below.
Answer with a single word: "Yes" or "No".

Context:
{context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

chain = (
    prompt
    | model
    | StrOutputParser()
)
runnable_parallel_object = RunnableParallel(context=retriever, question=RunnablePassthrough())
rag_chain = runnable_parallel_object.assign(answer=chain)

In [33]:
q = questions.iloc[0].question
out = rag_chain.invoke(q)

print("QUESTION:", out["question"])
print("RETRIEVED:", [d.metadata["id"] for d in out["context"]],
      "| GOLD:", questions.iloc[0].gold_document_id)
print("CONTEXT:", out["context"][0].page_content[:500])
print("ANSWER:", out["answer"])

QUESTION: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
RETRIEVED: [21645374, 21645374] | GOLD: 21645374
CONTEXT: Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has b
ANSWER:  No.


In [24]:
q = questions.iloc[0].question

docs = retriever.invoke(q)
print(type(docs), len(docs))
for d in docs:
    print(d.metadata, d.page_content[:300])

<class 'list'> 2
{'start_index': 0, 'id': 21645374} Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cel
{'start_index': 900, 'id': 21645374} of PCD (LPCD). Window stage leaves were stained with the mitochondrial dye MitoTracker Red CMXRos and examined. Mitochondrial dynamics were delineated into four categories (M1-M4) based on characteristics including distribution, motility, and membrane potential (ΔΨm). A TUNEL assay showed fragmented


In [26]:
docs

[Document(id='f2932095-74bc-49e5-b0fc-41456eb01304', metadata={'start_index': 0, 'id': 21645374}, page_content='Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in 

In [25]:
ctx_str = format_docs(docs)
print(ctx_str[:500])

Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has b


In [28]:
par_out = runnable_parallel_object.invoke(q)
print(par_out.keys())          # dict_keys(['context', 'question'])
print(type(par_out["context"]), par_out["question"])
par_out["context"]

dict_keys(['context', 'question'])
<class 'list'> Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?


[Document(id='f2932095-74bc-49e5-b0fc-41456eb01304', metadata={'id': 21645374, 'start_index': 0}, page_content='Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in 

In [32]:
prompt_val = prompt.invoke({"context": ctx_str, "question": q})
print(prompt_val.to_string())   # the exact text sent to the LM

Human: You are a biomedical expert. Answer the question using only the context below.
Answer with a single word: "Yes" or "No".

Context:
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of

In [30]:
raw = model.invoke(prompt_val)
print(repr(raw))   # repr shows leading spaces/newlines, useful for parsing

' No'


In [31]:
ans = StrOutputParser().invoke(raw)
print(ans)

 No


In [44]:
for i in range(len(questions)): 
    break
questions.iloc[i].question

'Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?'

In [49]:
answers = []
retrieved_document= []
for i in range(3): 

    q = questions.iloc[i]
    out = rag_chain.invoke(q.question)
    print("QUESTION:", out["question"])
    print("RETRIEVED:", [d.metadata["id"] for d in out["context"]],
          "| GOLD:", questions.iloc[0].gold_document_id)
    print("CONTEXT:", out["context"][0].page_content[:500])
    print("ANSWER:", out["answer"])
    answers.append(out["answer"])
    retrieved_document.append(q.gold_document_id in [d.metadata["id"] for d in out["context"]])

QUESTION: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
RETRIEVED: [21645374, 21645374] | GOLD: 21645374
CONTEXT: Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has b
ANSWER:  Yes.
QUESTION: Landolt C and snellen e acuity: differences in strabismus amblyopia?
RETRIEVED: [16418930, 16418930] | GOLD: 21645374
CONTEXT: acuity, and the right eyes of the healthy subjects, were evaluated. Differences between Landolt C acuity (LR) and Snellen E acuity (SE) were small. The mean decimal values for LR and SE wer

In [50]:
answers

[' Yes.', ' No', ' Yes']

In [51]:
retrieved_document

[True, True, True]

In [52]:
out

{'context': [Document(id='6fe989c8-a3de-4337-b8b2-472e3811fb37', metadata={'start_index': 898, 'id': 9488747}, page_content='tested. The evolution of these "aquagenic maladies" was favourable after a few weeks without baths. After a 2-7 year follow-up, three out of seven infants continue to suffer from troubles associated with sun or water. "Aquagenic maladies" could be a pediatric form of the aquagenic urticaria.'),
  Document(id='5536338a-9781-4821-9758-f7685ef6ebf3', metadata={'start_index': 0, 'id': 9488747}, page_content='Apparent life-threatening events in infants are a difficult and frequent problem in pediatric practice. The prognosis is uncertain because of risk of sudden infant death syndrome. Eight infants aged 2 to 15 months were admitted during a period of 6 years; they suffered from similar maladies in the bath: on immersion, they became pale, hypotonic, still and unreactive; recovery took a few seconds after withdrawal from the bath and stimulation. Two diagnoses were in

In [53]:
ans = StrOutputParser().invoke(out["answer"])
print(ans)

 Yes
